<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.10-ai-powered-ux/notebooks/exercises-10_10.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.10: Building AI-Powered UX

*Module 10 · AI System Architecture & Production Deployment*

> The backend is perfect. The model is great. But users hate the app because: it takes 5 seconds with a blank screen, errors show "500 Internal Server Error", and they can't tell when the AI is guessing. This lesson builds the frontend patterns that make AI apps feel responsive, transparent, and trustworthy.

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-10.10.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Streaming Text UI — First Token in 200ms — `streaming_backend.py`
2. Step 1 — Streaming Text UI — First Token in 200ms — `streaming_frontend.html`
3. Step 2 — Intelligent Loading States — Tell Users What's Happening — `loading_states.js`
4. Step 3 — Error Handling UX — Fail Gracefully, Recover Quickly — `error_handler.js`
5. Step 4 — Confidence & Attribution — Be Honest About Uncertainty — `confidence_ui.js`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai fastapi requests


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Streaming Text UI — First Token in 200ms

SSE backend + token-by-token frontend. Users see the response building in real time.

**`streaming_backend.py`** · _Block 1 of 5_

STREAMING BACKEND (Server-Sent Events) — Stream LLM tokens to the frontend in real time.


In [ ]:
# =============================================
# STREAMING BACKEND (Server-Sent Events)
# Stream LLM tokens to the frontend in real time.
# =============================================

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
import google.generativeai as genai
import json, os, time

genai.configure(api_key=os.getenv("GOOGLE_API_KEY", ""))
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

async def stream_llm(prompt: str):
    """Generator that yields SSE events token by token."""
    model = genai.GenerativeModel("gemini-2.0-flash")
    
    # Signal: streaming started
    yield f"data: {json.dumps({'type': 'start', 'timestamp': time.time()})}\n\n"
    
    # Stream tokens
    response = model.generate_content(prompt, stream=True)
    for chunk in response:
        if chunk.text:
            yield f"data: {json.dumps({'type': 'token', 'text': chunk.text})}\n\n"
    
    # Signal: streaming complete with metadata
    yield f"data: {json.dumps({'type': 'done', 'timestamp': time.time()})}\n\n"

@app.get("/v1/stream")
async def stream_chat(prompt: str):
    return StreamingResponse(
        stream_llm(prompt),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )


**`streaming_frontend.html`** · _Block 2 of 5_


In [ ]:
<!-- STREAMING FRONTEND — token-by-token display -->

<div id="chat-container">
  <div id="response" class="response-bubble"></div>
  <div id="cursor" class="typing-cursor"></div>
</div>

<script>
async function streamChat(prompt) {
  const responseEl = document.getElementById('response');
  const cursorEl = document.getElementById('cursor');
  responseEl.textContent = '';
  cursorEl.style.display = 'inline-block';

  const eventSource = new EventSource(
    `/v1/stream?prompt=${encodeURIComponent(prompt)}`
  );

  eventSource.onmessage = (event) => {
    const data = JSON.parse(event.data);

    if (data.type === 'token') {
      // Append token — this creates the "typing" effect
      responseEl.textContent += data.text;
      // Auto-scroll to bottom
      responseEl.scrollTop = responseEl.scrollHeight;
    }

    if (data.type === 'done') {
      cursorEl.style.display = 'none';
      eventSource.close();
    }
  };

  eventSource.onerror = () => {
    cursorEl.style.display = 'none';
    responseEl.textContent += '\n\n⚠️ Connection lost. Showing partial response.';
    eventSource.close();
  };
}
</script>

<style>
.response-bubble {
  max-height: 400px; overflow-y: auto; padding: 16px;
  background: #f8fafc; border-radius: 12px; font-size: 15px;
  line-height: 1.7; white-space: pre-wrap;
}
.typing-cursor {
  display: none; width: 8px; height: 18px;
  background: #4f46e5; border-radius: 2px;
  animation: blink 0.8s infinite;
}
@keyframes blink { 0%,50% { opacity: 1 } 51%,100% { opacity: 0 } }
</style>


> **What just happened?** Backend: FastAPI streams tokens via Server-Sent Events (SSE). Three event types: start (streaming began), token (each text chunk), done (complete with metadata). The X-Accel-Buffering: no header prevents nginx/Cloud Run from buffering. Frontend: EventSource listens for tokens and appends each one — creating the "typing" effect. A blinking cursor shows the AI is still generating. On error, the partial response is preserved with a warning. First token appears in ~200ms. The user reads while the AI thinks — perceived latency drops to near zero.


### Step 2 · Intelligent Loading States — Tell Users What's Happening

Not all waits are equal. A 2-second wait needs a spinner. A 15-second wait needs phases.

**`loading_states.js`** · _Block 3 of 5_


In [ ]:
// =============================================
// INTELLIGENT LOADING STATES
// Different UX for different wait durations.
// =============================================

class AILoadingState {
  constructor(containerEl) {
    this.container = containerEl;
    this.startTime = null;
    this.phaseTimer = null;
    this.abortController = null;
  }

  // Phase messages — rotate every few seconds for long waits
  phases = {
    "search":    ["Searching knowledge base...", "Reading relevant documents...", "Preparing answer..."],
    "generate":  ["Thinking...", "Drafting response...", "Refining answer..."],
    "analyze":   ["Analyzing your document...", "Extracting key points...", "Summarizing findings..."],
    "agent":     ["Planning approach...", "Using tools...", "Reviewing results...", "Composing answer..."],
  }

  show(taskType = "generate") {
    this.startTime = Date.now();
    this.abortController = new AbortController();
    const msgs = this.phases[taskType] || this.phases.generate;
    let phaseIdx = 0;

    this.container.innerHTML = `
      <div class="ai-loading">
        <div class="ai-loading-dots">
          <span></span><span></span><span></span>
        </div>
        <div class="ai-loading-text">${msgs[0]}</div>
        <div class="ai-loading-time"></div>
        <button class="ai-loading-cancel">Cancel</button>
      </div>
    `;

    // Cancel button
    this.container.querySelector('.ai-loading-cancel')
      .addEventListener('click', () => this.cancel());

    // Rotate phase messages every 3 seconds
    this.phaseTimer = setInterval(() => {
      phaseIdx = (phaseIdx + 1) % msgs.length;
      const textEl = this.container.querySelector('.ai-loading-text');
      if (textEl) textEl.textContent = msgs[phaseIdx];

      // Show elapsed time after 5 seconds
      const elapsed = Math.round((Date.now() - this.startTime) / 1000);
      if (elapsed >= 5) {
        const timeEl = this.container.querySelector('.ai-loading-time');
        if (timeEl) timeEl.textContent = `${elapsed}s elapsed`;
      }
    }, 3000);
  }

  hide() {
    clearInterval(this.phaseTimer);
    this.container.innerHTML = '';
  }

  cancel() {
    if (this.abortController) this.abortController.abort();
    this.hide();
    this.container.innerHTML = '<div class="ai-cancelled">Request cancelled.</div>';
  }
}

// Usage:
// const loader = new AILoadingState(document.getElementById('chat'));
// loader.show("agent");   // shows "Planning approach..." → "Using tools..." → etc.
// ... await fetch() ...
// loader.hide();


> **What just happened?** AILoadingState shows context-aware progress: search ("Searching knowledge base..." → "Reading documents..." → "Preparing answer..."), generate ("Thinking..." → "Drafting..."), agent ("Planning approach..." → "Using tools..." → "Reviewing results..."). Messages rotate every 3 seconds. After 5 seconds, elapsed time is shown. A cancel button lets users abort long requests. Users who see "Searching knowledge base..." wait patiently. Users who see a blank spinner for 10 seconds leave.


### Step 3 · Error Handling UX — Fail Gracefully, Recover Quickly

No user should ever see "500 Internal Server Error." Every error has a friendly face.

**`error_handler.js`** · _Block 4 of 5_


In [ ]:
// =============================================
// AI ERROR HANDLING UX
// Map API errors to user-friendly messages
// with recovery actions.
// =============================================

const ERROR_MAP = {
  // HTTP status → user message + recovery action
  401: {
    title: "Session expired",
    message: "Please log in again to continue.",
    action: { label: "Log in", handler: () => window.location.href = "/login" },
    icon: "🔒",
  },
  429: {
    title: "Too many requests",
    message: "You're sending requests too quickly. Please wait a moment.",
    action: { label: "Try again", handler: null, autoRetry: true, delayMs: 5000 },
    icon: "⏳",
  },
  402: {
    title: "Usage limit reached",
    message: "You've reached your daily limit. Upgrade for more.",
    action: { label: "View plans", handler: () => window.location.href = "/pricing" },
    icon: "📊",
  },
  500: {
    title: "Something went wrong",
    message: "Our AI is having a moment. We're working on it.",
    action: { label: "Try again", handler: null, autoRetry: true, delayMs: 3000 },
    icon: "🔧",
  },
  503: {
    title: "AI is busy",
    message: "High demand right now. Your request is queued.",
    action: { label: "Wait", handler: null, autoRetry: true, delayMs: 10000 },
    icon: "🔄",
  },
  timeout: {
    title: "Taking too long",
    message: "The AI is thinking hard about this one. Try a simpler question?",
    action: { label: "Try again", handler: null },
    secondaryAction: { label: "Simplify question", handler: null },
    icon: "⏱️",
  },
  network: {
    title: "No connection",
    message: "Check your internet connection and try again.",
    action: { label: "Retry", handler: null, autoRetry: true, delayMs: 5000 },
    icon: "📡",
  },
};

function showError(containerEl, errorCode, retryFn) {
  const error = ERROR_MAP[errorCode] || ERROR_MAP[500];

  containerEl.innerHTML = `
    <div class="ai-error">
      <span class="ai-error-icon">${error.icon}</span>
      <div class="ai-error-title">${error.title}</div>
      <div class="ai-error-msg">${error.message}</div>
      <div class="ai-error-actions">
        <button class="ai-error-btn primary">${error.action.label}</button>
        ${error.secondaryAction
          ? \`<button class="ai-error-btn secondary">${error.secondaryAction.label}</button>\`
          : ''}
      </div>
      ${error.action.autoRetry
        ? \`<div class="ai-error-auto">Auto-retrying in ${error.action.delayMs/1000}s...</div>\`
        : ''}
    </div>
  `;

  // Bind retry
  const btn = containerEl.querySelector('.primary');
  btn.addEventListener('click', () => {
    if (error.action.handler) error.action.handler();
    else if (retryFn) retryFn();
  });

  // Auto-retry for transient errors
  if (error.action.autoRetry && retryFn) {
    setTimeout(() => retryFn(), error.action.delayMs);
  }
}


> **What just happened?** Every API error mapped to a user-friendly experience: 429 → "Too many requests" + auto-retry in 5s. 402 → "Usage limit reached" + "View plans" button. 500 → "Our AI is having a moment" + auto-retry in 3s. 503 → "AI is busy, you're queued" + auto-retry in 10s. timeout → "Try a simpler question?" + two action buttons. network → "Check your connection" + auto-retry. No user ever sees a raw HTTP error code. Auto-retry handles transient errors without user action.


### Step 4 · Confidence & Attribution — Be Honest About Uncertainty

Show users when the AI is sure vs guessing. Cite sources. Build trust through transparency.

**`confidence_ui.js`** · _Block 5 of 5_


In [ ]:
// =============================================
// CONFIDENCE INDICATORS & SOURCE ATTRIBUTION
// Show users how certain the AI is and where
// the information comes from.
// =============================================

function renderAIResponse(response) {
  /*
    response = {
      text: "MCP is an open standard...",
      confidence: 0.92,          // 0-1 from the backend
      sources: [                 // RAG sources used
        { title: "Anthropic Blog", url: "...", relevance: 0.95 },
        { title: "MCP Spec", url: "...", relevance: 0.88 },
      ],
      cached: false,
      model: "gemini-2.0-flash",
      latency_ms: 845,
    }
  */

  const conf = response.confidence;

  // Confidence badge
  let confBadge;
  if (conf >= 0.9) {
    confBadge = `<span class="conf-badge conf-high">High confidence</span>`;
  } else if (conf >= 0.7) {
    confBadge = `<span class="conf-badge conf-med">Moderate confidence</span>`;
  } else {
    confBadge = `<span class="conf-badge conf-low">Low confidence — verify this answer</span>`;
  }

  // Source pills
  const sources = (response.sources || [])
    .map(s => `<a href="${s.url}" class="source-pill" target="_blank">
      ${s.title} <span class="source-rel">${Math.round(s.relevance*100)}%</span>
    </a>`)
    .join('');

  // Disclaimer for low confidence
  const disclaimer = conf < 0.7
    ? `<div class="ai-disclaimer">
        ⚠️ This answer may be incomplete or inaccurate.
        Consider verifying with official sources.
       </div>`
    : '';

  // Metadata footer
  const meta = `<div class="ai-meta">
    ${confBadge}
    ${response.cached ? '<span class="meta-tag">Cached</span>' : ''}
    <span class="meta-tag">${response.latency_ms}ms</span>
  </div>`;

  return `
    <div class="ai-response">
      <div class="ai-response-text">${response.text}</div>
      ${disclaimer}
      ${sources ? \`<div class="ai-sources">Sources: ${sources}</div>\` : ''}
      ${meta}
    </div>
  `;
}

// ── CSS (inline for portability) ──
const STYLES = `
.conf-badge { font-size: 11px; padding: 2px 8px; border-radius: 12px; font-weight: 600; }
.conf-high  { background: #d1fae5; color: #065f46; }
.conf-med   { background: #fef3c7; color: #92400e; }
.conf-low   { background: #ffe4e6; color: #9f1239; }
.source-pill { display: inline-flex; align-items: center; gap: 4px; padding: 3px 10px;
  background: #f1f5f9; border: 1px solid #e2e8f0; border-radius: 16px; font-size: 12px;
  color: #334155; text-decoration: none; margin: 2px; }
.source-pill:hover { background: #e2e8f0; }
.source-rel { font-size: 10px; color: #94a3b8; }
.ai-disclaimer { background: #fffbeb; border: 1px solid #fef08a; border-radius: 8px;
  padding: 8px 12px; font-size: 13px; color: #92400e; margin: 10px 0; }
.ai-meta { display: flex; gap: 6px; flex-wrap: wrap; margin-top: 10px; }
.meta-tag { font-size: 11px; color: #94a3b8; background: #f8fafc; padding: 2px 8px;
  border-radius: 8px; border: 1px solid #e2e8f0; }
.ai-sources { margin-top: 10px; font-size: 12px; color: #64748b; }
`;


> **What just happened?** renderAIResponse() adds three trust signals: (1) Confidence badge — green "High confidence" (≥90%), amber "Moderate" (70-90%), red "Low — verify this answer" (<70%). (2) Source pills — clickable links to RAG sources with relevance percentages ("Anthropic Blog 95%"). (3) Disclaimer — amber warning box on low-confidence answers: "This answer may be incomplete. Verify with official sources." Metadata footer shows: cached status, latency. Users who see "High confidence" + 2 sources trust the answer. Users who see "Low confidence — verify" know to double-check. Transparency builds trust.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
